# Temporal Emotion Analysis

This notebook demonstrates advanced temporal analysis of emotions, including:

1. Emotion changes and transitions over time
2. Temporal smoothing techniques
3. Microexpression detection
4. Emotion stability analysis
5. Statistical analysis of emotion sequences
6. Transition matrices and patterns

## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from scipy import signal
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from asdrp import (
    MediaPipeFaceDetector, VideoFileReader,
    GeometryBasedEmotionAnalyzer, EmotionType,
    TemporalEmotionAnalyzer
)
from asdrp.emotion.metrics import (
    compute_emotion_distribution,
    detect_emotion_transitions,
    compute_emotion_stability,
    compute_confidence_statistics
)

# Configure plotting
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid")

print("Setup complete!")

## 1. Load and Process Video

Let's process the video with and without temporal smoothing to compare the results.

In [ ]:
# Paths
video_path = project_root / "data" / "videos" / "youtube_short_emotion.mp4"
model_path = project_root / "models" / "face_landmarker.task"

# Check files
if not video_path.exists():
    print(f"Error: Video not found at {video_path}")
if not model_path.exists():
    print(f"Error: Model not found at {model_path}")
else:
    print(f"✓ Files ready")

In [ ]:
# Process video with different smoothing configurations
if model_path.exists():
    # Initialize components
    face_detector = MediaPipeFaceDetector(
        model_path=str(model_path),
        min_detection_confidence=0.5,
        num_faces=1,
        running_mode="VIDEO"
    )
    
    emotion_analyzer = GeometryBasedEmotionAnalyzer(confidence_threshold=0.3)
    
    # Create temporal analyzers with different window sizes
    temporal_no_smooth = TemporalEmotionAnalyzer(window_size=1)  # No smoothing
    temporal_small = TemporalEmotionAnalyzer(window_size=3)      # Small window
    temporal_medium = TemporalEmotionAnalyzer(window_size=7)     # Medium window
    temporal_large = TemporalEmotionAnalyzer(window_size=15)     # Large window
    
    print("✓ Components initialized")
    
    # Process video
    with VideoFileReader(str(video_path)) as reader:
        metadata = reader.metadata
        print(f"\nProcessing: {metadata.width}x{metadata.height} @ {metadata.fps:.2f} FPS")
        
        # Process frames
        skip_frames = 1
        frames_to_process = list(range(0, min(metadata.total_frames, 400), skip_frames + 1))
        
        results = []
        print(f"Processing {len(frames_to_process)} frames...\n")
        
        for i, frame_num in enumerate(frames_to_process):
            frame_data = reader.get_frame_at(frame_num)
            if frame_data is None:
                continue
            
            # Detect face
            faces = face_detector.detect(frame_data.frame, timestamp_ms=frame_data.timestamp_ms)
            
            if faces:
                face = faces[0]
                
                # Analyze emotion
                raw_prediction = emotion_analyzer.analyze(face)
                
                # Apply different smoothing levels
                no_smooth = temporal_no_smooth.smooth_prediction(raw_prediction)
                small_smooth = temporal_small.smooth_prediction(raw_prediction)
                medium_smooth = temporal_medium.smooth_prediction(raw_prediction)
                large_smooth = temporal_large.smooth_prediction(raw_prediction)
                
                results.append({
                    'frame_number': frame_num,
                    'timestamp_ms': frame_data.timestamp_ms,
                    'timestamp_s': frame_data.timestamp_ms / 1000,
                    'raw': raw_prediction,
                    'no_smooth': no_smooth,
                    'small_smooth': small_smooth,
                    'medium_smooth': medium_smooth,
                    'large_smooth': large_smooth
                })
            
            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1}/{len(frames_to_process)} frames")
        
        print(f"\n✓ Completed! Processed {len(results)} frames")

## 2. Temporal Smoothing Comparison

Let's visualize how different smoothing window sizes affect the emotion predictions.

In [ ]:
# Compare smoothing effects
if results:
    timestamps = [r['timestamp_s'] for r in results]
    
    # Map emotions to numeric values
    emotion_types = list(EmotionType)
    emotion_to_num = {e: i for i, e in enumerate(emotion_types)}
    
    # Extract emotion sequences for different smoothing levels
    smoothing_levels = ['no_smooth', 'small_smooth', 'medium_smooth', 'large_smooth']
    level_names = ['No Smoothing (Window=1)', 'Small (Window=3)', 
                   'Medium (Window=7)', 'Large (Window=15)']
    
    fig, axes = plt.subplots(len(smoothing_levels), 1, figsize=(18, 14), sharex=True)
    
    colors_map = {
        EmotionType.NEUTRAL: '#C0C0C0',
        EmotionType.HAPPY: '#FFD700',
        EmotionType.SAD: '#4682B4',
        EmotionType.ANGRY: '#DC143C',
        EmotionType.SURPRISED: '#FF8C00',
        EmotionType.FEARFUL: '#9370DB',
        EmotionType.DISGUSTED: '#228B22'
    }
    
    for idx, (level, name) in enumerate(zip(smoothing_levels, level_names)):
        emotions = [r[level].emotion for r in results]
        emotion_values = [emotion_to_num[e] for e in emotions]
        
        # Plot with colors
        for i in range(len(timestamps) - 1):
            color = colors_map.get(emotions[i], '#808080')
            axes[idx].plot(timestamps[i:i+2], emotion_values[i:i+2],
                         color=color, linewidth=2.5, marker='o', markersize=3)
        
        axes[idx].set_ylabel('Emotion', fontsize=11, fontweight='bold')
        axes[idx].set_yticks(range(len(emotion_types)))
        axes[idx].set_yticklabels([e.value.capitalize() for e in emotion_types], fontsize=9)
        axes[idx].set_title(name, fontsize=12, fontweight='bold', loc='left')
        axes[idx].grid(True, alpha=0.3)
        
        # Count transitions
        transitions = sum(1 for i in range(len(emotions)-1) if emotions[i] != emotions[i+1])
        axes[idx].text(0.98, 0.95, f'Transitions: {transitions}', 
                      transform=axes[idx].transAxes,
                      ha='right', va='top', fontsize=10,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    axes[-1].set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    plt.suptitle('Effect of Temporal Smoothing on Emotion Detection', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\nSmoothing Effect Statistics:")
    print("=" * 70)
    for level, name in zip(smoothing_levels, level_names):
        emotions = [r[level].emotion for r in results]
        transitions = sum(1 for i in range(len(emotions)-1) if emotions[i] != emotions[i+1])
        unique_emotions = len(set(emotions))
        
        print(f"\n{name}:")
        print(f"  Total transitions: {transitions}")
        print(f"  Transition rate: {transitions/(len(emotions)-1)*100:.1f}%")
        print(f"  Unique emotions: {unique_emotions}")
        print(f"  Most common: {Counter(emotions).most_common(1)[0]}")

## 3. Emotion Stability Analysis

Let's analyze how stable emotions are over time using various metrics.

In [ ]:
# Emotion stability metrics
if results:
    # Use medium smoothing for analysis
    predictions = [r['medium_smooth'] for r in results]
    
    # Compute stability metrics
    stability = compute_emotion_stability(predictions)
    
    print("Emotion Stability Metrics:")
    print("=" * 70)
    print(f"\nOverall Stability Score: {stability['overall_stability']:.3f}")
    print(f"  (0 = very unstable, 1 = very stable)\n")
    
    print(f"Transition Rate: {stability['transition_rate']:.3f}")
    print(f"  ({stability['num_transitions']} transitions in {len(predictions)-1} frames)\n")
    
    print(f"Average Duration: {stability['average_duration']:.2f} frames")
    print(f"  (average time each emotion persists)\n")
    
    print(f"Confidence Stability: {stability['confidence_stability']:.3f}")
    print(f"  (lower std = more stable confidence)\n")
    
    # Duration distribution
    if stability['duration_distribution']:
        print("\nEmotion Duration Distribution:")
        for emotion, durations in sorted(stability['duration_distribution'].items()):
            if durations:
                avg_dur = np.mean(durations)
                max_dur = np.max(durations)
                print(f"  {emotion.value.capitalize():12s}: avg={avg_dur:5.1f} frames, "
                      f"max={max_dur:3d} frames, occurrences={len(durations)}")

In [ ]:
# Visualize stability metrics
if results:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Confidence over time with variance bands
    confidences = [r['medium_smooth'].confidence for r in results]
    timestamps = [r['timestamp_s'] for r in results]
    
    # Calculate rolling statistics
    window = 10
    conf_series = pd.Series(confidences)
    rolling_mean = conf_series.rolling(window=window, center=True).mean()
    rolling_std = conf_series.rolling(window=window, center=True).std()
    
    axes[0, 0].plot(timestamps, confidences, alpha=0.4, linewidth=1, label='Raw', color='gray')
    axes[0, 0].plot(timestamps, rolling_mean, linewidth=2, label=f'Rolling Mean (w={window})', 
                   color='blue')
    axes[0, 0].fill_between(timestamps, 
                           rolling_mean - rolling_std, 
                           rolling_mean + rolling_std,
                           alpha=0.3, color='blue', label='±1 Std Dev')
    axes[0, 0].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Confidence', fontsize=11, fontweight='bold')
    axes[0, 0].set_title('Confidence Stability Over Time', fontsize=12, fontweight='bold')
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Emotion duration distribution
    if stability['duration_distribution']:
        emotion_labels = []
        durations_data = []
        
        for emotion, durations in sorted(stability['duration_distribution'].items()):
            if durations:
                emotion_labels.append(emotion.value.capitalize())
                durations_data.append(durations)
        
        bp = axes[0, 1].boxplot(durations_data, labels=emotion_labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], ['#C0C0C0', '#FFD700', '#4682B4', 
                                               '#DC143C', '#FF8C00', '#9370DB', '#228B22']):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        axes[0, 1].set_xlabel('Emotion', fontsize=11, fontweight='bold')
        axes[0, 1].set_ylabel('Duration (frames)', fontsize=11, fontweight='bold')
        axes[0, 1].set_title('Emotion Duration Distribution', fontsize=12, fontweight='bold')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Transition frequency over time
    emotions = [r['medium_smooth'].emotion for r in results]
    transitions = [1 if i > 0 and emotions[i] != emotions[i-1] else 0 
                  for i in range(len(emotions))]
    
    # Compute rolling transition rate
    trans_window = 20
    trans_series = pd.Series(transitions)
    rolling_trans = trans_series.rolling(window=trans_window, center=True).sum()
    
    axes[1, 0].plot(timestamps, rolling_trans, linewidth=2, color='red')
    axes[1, 0].fill_between(timestamps, rolling_trans, alpha=0.3, color='red')
    axes[1, 0].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel(f'Transitions (rolling {trans_window}-frame window)', 
                         fontsize=11, fontweight='bold')
    axes[1, 0].set_title('Emotion Transition Frequency Over Time', 
                        fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Entropy over time (emotion uncertainty)
    entropies = []
    for result in results:
        probs = result['medium_smooth'].probabilities
        prob_values = [probs.get(e, 0.0) for e in EmotionType]
        # Add small epsilon to avoid log(0)
        prob_values = np.array(prob_values) + 1e-10
        prob_values = prob_values / prob_values.sum()  # Renormalize
        ent = entropy(prob_values, base=2)  # Bits of entropy
        entropies.append(ent)
    
    ent_series = pd.Series(entropies)
    rolling_ent = ent_series.rolling(window=10, center=True).mean()
    
    axes[1, 1].plot(timestamps, entropies, alpha=0.3, linewidth=1, label='Raw', color='gray')
    axes[1, 1].plot(timestamps, rolling_ent, linewidth=2, label='Rolling Mean', color='purple')
    axes[1, 1].set_xlabel('Time (s)', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Entropy (bits)', fontsize=11, fontweight='bold')
    axes[1, 1].set_title('Emotion Uncertainty Over Time\n(Higher = More Uncertain)', 
                        fontsize=12, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle('Emotion Stability Analysis', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

## 4. Emotion Transition Analysis

Let's analyze which emotions transition to which other emotions.

In [ ]:
# Detect and analyze transitions
if results:
    predictions = [r['medium_smooth'] for r in results]
    transitions = detect_emotion_transitions(predictions)
    
    print("Emotion Transitions:")
    print("=" * 70)
    print(f"\nTotal transitions detected: {len(transitions)}\n")
    
    # Show first few transitions
    print("First 10 transitions:")
    for i, trans in enumerate(transitions[:10]):
        print(f"  {i+1}. Frame {trans['frame_number']}: "
              f"{trans['from_emotion'].value} → {trans['to_emotion'].value} "
              f"(duration: {trans['duration']} frames)")

In [ ]:
# Create transition matrix
if results and transitions:
    # Build transition matrix
    emotion_types = list(EmotionType)
    n = len(emotion_types)
    transition_matrix = np.zeros((n, n))
    
    for trans in transitions:
        from_idx = emotion_types.index(trans['from_emotion'])
        to_idx = emotion_types.index(trans['to_emotion'])
        transition_matrix[from_idx, to_idx] += 1
    
    # Normalize rows to get probabilities
    row_sums = transition_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    transition_probs = transition_matrix / row_sums
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    emotion_labels = [e.value.capitalize() for e in emotion_types]
    
    # Counts
    sns.heatmap(transition_matrix, annot=True, fmt='.0f', cmap='YlOrRd',
               xticklabels=emotion_labels, yticklabels=emotion_labels,
               ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_xlabel('To Emotion', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('From Emotion', fontsize=12, fontweight='bold')
    axes[0].set_title('Emotion Transition Count Matrix', fontsize=14, fontweight='bold')
    
    # Probabilities
    sns.heatmap(transition_probs, annot=True, fmt='.2f', cmap='Blues',
               xticklabels=emotion_labels, yticklabels=emotion_labels,
               ax=axes[1], cbar_kws={'label': 'Probability'}, vmin=0, vmax=1)
    axes[1].set_xlabel('To Emotion', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('From Emotion', fontsize=12, fontweight='bold')
    axes[1].set_title('Emotion Transition Probability Matrix', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Find most common transitions
    print("\nMost Common Transitions:")
    print("=" * 70)
    
    # Get all non-zero transitions
    transition_list = []
    for i in range(n):
        for j in range(n):
            if transition_matrix[i, j] > 0:
                transition_list.append((
                    emotion_types[i],
                    emotion_types[j],
                    int(transition_matrix[i, j]),
                    transition_probs[i, j]
                ))
    
    # Sort by count
    transition_list.sort(key=lambda x: x[2], reverse=True)
    
    for from_emo, to_emo, count, prob in transition_list[:10]:
        print(f"  {from_emo.value:12s} → {to_emo.value:12s}: "
              f"{count:3d} times ({prob*100:5.1f}%)")

## 5. Microexpression Detection

Microexpressions are brief, involuntary facial expressions that last only a fraction of a second. Let's detect potential microexpressions in the video.

In [ ]:
# Detect microexpressions
if results and model_path.exists():
    # Microexpressions are typically:
    # 1. Very brief (1/25 to 1/5 second = 2-12 frames at 30fps)
    # 2. High confidence
    # 3. Different from surrounding emotions
    
    microexpressions = []
    emotions = [r['no_smooth'].emotion for r in results]  # Use raw, unsmoothed
    confidences = [r['no_smooth'].confidence for r in results]
    
    min_duration = 2   # At least 2 frames
    max_duration = 12  # At most 12 frames
    min_confidence = 0.6
    
    i = 0
    while i < len(emotions):
        current_emotion = emotions[i]
        
        # Find duration of this emotion
        duration = 1
        j = i + 1
        while j < len(emotions) and emotions[j] == current_emotion:
            duration += 1
            j += 1
        
        # Check if it's a microexpression
        if min_duration <= duration <= max_duration:
            # Check if confidence is high enough
            avg_confidence = np.mean(confidences[i:j])
            
            if avg_confidence >= min_confidence:
                # Check if surrounded by different emotions
                before_emotion = emotions[i-1] if i > 0 else None
                after_emotion = emotions[j] if j < len(emotions) else None
                
                if (before_emotion != current_emotion or after_emotion != current_emotion):
                    microexpressions.append({
                        'start_frame': results[i]['frame_number'],
                        'end_frame': results[j-1]['frame_number'],
                        'start_time': results[i]['timestamp_s'],
                        'end_time': results[j-1]['timestamp_s'],
                        'duration_frames': duration,
                        'emotion': current_emotion,
                        'confidence': avg_confidence,
                        'before': before_emotion,
                        'after': after_emotion
                    })
        
        i = j
    
    print(f"Detected {len(microexpressions)} potential microexpressions:")
    print("=" * 80)
    
    for idx, micro in enumerate(microexpressions[:15]):  # Show first 15
        print(f"\n{idx+1}. {micro['emotion'].value.upper()}")
        print(f"   Time: {micro['start_time']:.2f}s - {micro['end_time']:.2f}s "
              f"({micro['duration_frames']} frames)")
        print(f"   Confidence: {micro['confidence']:.3f}")
        print(f"   Context: {micro['before'].value if micro['before'] else 'START'} → "
              f"{micro['emotion'].value} → {micro['after'].value if micro['after'] else 'END'}")
    
    if len(microexpressions) > 15:
        print(f"\n... and {len(microexpressions) - 15} more")

In [ ]:
# Visualize microexpressions on timeline
if results and microexpressions:
    fig, ax = plt.subplots(figsize=(18, 6))
    
    # Plot full emotion timeline
    timestamps = [r['timestamp_s'] for r in results]
    emotions = [r['medium_smooth'].emotion for r in results]
    emotion_types = list(EmotionType)
    emotion_to_num = {e: i for i, e in enumerate(emotion_types)}
    emotion_values = [emotion_to_num[e] for e in emotions]
    
    ax.plot(timestamps, emotion_values, color='gray', linewidth=1, alpha=0.5, 
           label='Smoothed emotions')
    
    # Highlight microexpressions
    for micro in microexpressions:
        ax.axvspan(micro['start_time'], micro['end_time'], 
                  alpha=0.3, color='red', linewidth=0)
    
    # Add one legend entry for microexpressions
    ax.axvspan(0, 0, alpha=0.3, color='red', label='Microexpression')
    
    ax.set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Emotion', fontsize=12, fontweight='bold')
    ax.set_yticks(range(len(emotion_types)))
    ax.set_yticklabels([e.value.capitalize() for e in emotion_types], fontsize=10)
    ax.set_title(f'Microexpression Detection ({len(microexpressions)} detected)', 
                fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Statistical Analysis

Let's perform comprehensive statistical analysis of the emotion sequences.

In [ ]:
# Comprehensive statistical analysis
if results:
    predictions = [r['medium_smooth'] for r in results]
    
    # Compute various metrics
    distribution = compute_emotion_distribution(predictions)
    conf_stats = compute_confidence_statistics(predictions)
    
    # Create comprehensive report
    print("="*80)
    print("COMPREHENSIVE EMOTION ANALYSIS REPORT")
    print("="*80)
    
    print("\n1. BASIC STATISTICS")
    print("-" * 80)
    print(f"Total frames analyzed: {len(results)}")
    print(f"Video duration: {results[-1]['timestamp_s']:.2f} seconds")
    print(f"Average frame rate: {len(results)/results[-1]['timestamp_s']:.2f} FPS")
    
    print("\n2. EMOTION DISTRIBUTION")
    print("-" * 80)
    for emotion, info in sorted(distribution.items(), 
                                key=lambda x: x[1]['count'], reverse=True):
        print(f"{emotion.value.capitalize():12s}: {info['count']:4d} frames "
              f"({info['percentage']:5.1f}%) | "
              f"Avg conf: {info['average_confidence']:.3f}")
    
    print("\n3. CONFIDENCE STATISTICS")
    print("-" * 80)
    print(f"Mean confidence:     {conf_stats['mean']:.3f}")
    print(f"Median confidence:   {conf_stats['median']:.3f}")
    print(f"Std deviation:       {conf_stats['std']:.3f}")
    print(f"Min confidence:      {conf_stats['min']:.3f}")
    print(f"Max confidence:      {conf_stats['max']:.3f}")
    print(f"25th percentile:     {conf_stats['percentile_25']:.3f}")
    print(f"75th percentile:     {conf_stats['percentile_75']:.3f}")
    
    print("\n4. TEMPORAL PATTERNS")
    print("-" * 80)
    print(f"Total transitions:   {stability['num_transitions']}")
    print(f"Transition rate:     {stability['transition_rate']*100:.1f}%")
    print(f"Average duration:    {stability['average_duration']:.1f} frames")
    print(f"Stability score:     {stability['overall_stability']:.3f}")
    
    if microexpressions:
        print("\n5. MICROEXPRESSIONS")
        print("-" * 80)
        print(f"Total detected:      {len(microexpressions)}")
        micro_emotions = Counter([m['emotion'] for m in microexpressions])
        print("\nBy emotion:")
        for emotion, count in micro_emotions.most_common():
            print(f"  {emotion.value.capitalize():12s}: {count:2d}")
    
    print("\n" + "="*80)

## 7. Advanced Visualizations

Let's create some advanced visualizations combining multiple aspects of temporal analysis.

In [ ]:
# Combined temporal analysis visualization
if results:
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    timestamps = [r['timestamp_s'] for r in results]
    emotions = [r['medium_smooth'].emotion for r in results]
    confidences = [r['medium_smooth'].confidence for r in results]
    
    emotion_types = list(EmotionType)
    emotion_to_num = {e: i for i, e in enumerate(emotion_types)}
    emotion_values = [emotion_to_num[e] for e in emotions]
    
    # 1. Main timeline
    ax1 = fig.add_subplot(gs[0, :])
    
    colors_map = {
        EmotionType.NEUTRAL: '#C0C0C0',
        EmotionType.HAPPY: '#FFD700',
        EmotionType.SAD: '#4682B4',
        EmotionType.ANGRY: '#DC143C',
        EmotionType.SURPRISED: '#FF8C00',
        EmotionType.FEARFUL: '#9370DB',
        EmotionType.DISGUSTED: '#228B22'
    }
    
    for i in range(len(timestamps) - 1):
        color = colors_map.get(emotions[i], '#808080')
        ax1.plot(timestamps[i:i+2], emotion_values[i:i+2],
                color=color, linewidth=3)
    
    # Highlight transitions
    for i in range(1, len(emotions)):
        if emotions[i] != emotions[i-1]:
            ax1.axvline(timestamps[i], color='red', alpha=0.3, linewidth=1, linestyle='--')
    
    ax1.set_ylabel('Emotion', fontsize=11, fontweight='bold')
    ax1.set_yticks(range(len(emotion_types)))
    ax1.set_yticklabels([e.value.capitalize() for e in emotion_types], fontsize=10)
    ax1.set_title('Emotion Timeline with Transitions', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 2. Confidence with annotations
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(timestamps, confidences, linewidth=2, color='#2E86AB')
    ax2.fill_between(timestamps, confidences, alpha=0.3, color='#2E86AB')
    ax2.axhline(y=np.mean(confidences), color='green', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(confidences):.3f}')
    ax2.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Confidence', fontsize=11, fontweight='bold')
    ax2.set_title('Confidence Score', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # 3. Emotion distribution pie chart
    ax3 = fig.add_subplot(gs[1, 1])
    emotion_counts = Counter(emotions)
    labels = [e.value.capitalize() for e in emotion_counts.keys()]
    sizes = list(emotion_counts.values())
    colors = [colors_map.get(e, '#808080') for e in emotion_counts.keys()]
    
    wedges, texts, autotexts = ax3.pie(sizes, labels=labels, autopct='%1.1f%%',
                                       colors=colors, startangle=90)
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(10)
    ax3.set_title('Emotion Distribution', fontsize=13, fontweight='bold')
    
    # 4. Cumulative emotion duration
    ax4 = fig.add_subplot(gs[2, 0])
    
    # Calculate cumulative durations
    cumulative_data = {e: np.zeros(len(timestamps)) for e in EmotionType}
    for i, (timestamp, emotion) in enumerate(zip(timestamps, emotions)):
        for e in EmotionType:
            if i > 0:
                cumulative_data[e][i] = cumulative_data[e][i-1]
            if emotion == e:
                cumulative_data[e][i] += 1
    
    # Plot top emotions
    top_emotions = sorted(emotion_counts.keys(), key=lambda e: emotion_counts[e], reverse=True)[:4]
    for emotion in top_emotions:
        ax4.plot(timestamps, cumulative_data[emotion], 
                linewidth=2, label=emotion.value.capitalize(),
                color=colors_map.get(emotion, '#808080'))
    
    ax4.set_xlabel('Time (s)', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Cumulative Frame Count', fontsize=11, fontweight='bold')
    ax4.set_title('Cumulative Emotion Duration', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3)
    
    # 5. Summary statistics
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    
    summary_text = f"""
    SUMMARY STATISTICS
    
    Video Duration:        {results[-1]['timestamp_s']:.2f} seconds
    Total Frames:          {len(results)}
    
    Emotion Changes:       {stability['num_transitions']}
    Avg. Duration:         {stability['average_duration']:.1f} frames
    Stability Score:       {stability['overall_stability']:.3f}
    
    Confidence Mean:       {np.mean(confidences):.3f}
    Confidence Std:        {np.std(confidences):.3f}
    
    Dominant Emotion:      {max(emotion_counts, key=emotion_counts.get).value.capitalize()}
    ({emotion_counts[max(emotion_counts, key=emotion_counts.get)]} frames)
    
    Unique Emotions:       {len(emotion_counts)}
    """
    
    ax5.text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Comprehensive Temporal Emotion Analysis', 
                fontsize=18, fontweight='bold', y=0.995)
    plt.show()

## Summary

In this notebook, we explored:

1. **Temporal Smoothing**: Different window sizes and their effects on emotion stability
2. **Emotion Stability**: Metrics for measuring how consistent emotions are over time
3. **Transition Analysis**: Understanding which emotions follow which others
4. **Microexpressions**: Detecting brief, involuntary emotional displays
5. **Statistical Analysis**: Comprehensive metrics for emotion sequences
6. **Advanced Visualizations**: Multi-faceted views of temporal patterns

### Key Insights

- Temporal smoothing reduces noise but may miss rapid emotional changes
- Emotion transitions reveal patterns in emotional expression
- Microexpressions can indicate suppressed or genuine emotions
- Stability metrics help assess the reliability of emotion detection

### Applications

- **Video Analysis**: Understanding emotional arcs in videos
- **Human-Computer Interaction**: Responding to user emotional states
- **Psychology Research**: Studying emotional expression patterns
- **Content Analysis**: Assessing emotional impact of media

### Clean Up

In [ ]:
# Clean up
if model_path.exists():
    face_detector.close()
    print("Resources cleaned up successfully!")